# 03 - MongoDB Atlas and Optimisation

This notebook covers the MongoDB Atlas and query optimisation parts of the MedLine assignment. It uses the nested patient service cases, imports them into MongoDB, demonstrates CRUD, runs aggregation queries, creates indexes, and records timing/explain evidence.

## 1. Setup

This cell installs and imports the Python libraries, finds the dataset folder and prepares output folders. The MongoDB connection string must be stored privately as `MONGODB_URI`.

In [1]:
import sys
import subprocess
from pathlib import Path
import json
import os
import time

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "pymongo", "dnspython"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

import pandas as pd
from IPython.display import display
from pymongo import MongoClient, ASCENDING, DESCENDING

ROOT = Path.cwd()
possible_data_dirs = [
    ROOT / "medline_resit_dataset",
    ROOT / "data",
    ROOT.parent / "medline_resit_dataset",
    ROOT.parent / "data",
]
DATA = next((path for path in possible_data_dirs if (path / "patient_service_cases.jsonl").exists()), None)
if DATA is None:
    raise FileNotFoundError("Dataset folder not found. Upload medline_resit_dataset or data folder.")

PROJECT_ROOT = DATA.parent
OUT = PROJECT_ROOT / "outputs"
TABLES = OUT / "tables"
MONGO_OUT = OUT / "mongodb"
TABLES.mkdir(parents=True, exist_ok=True)
MONGO_OUT.mkdir(parents=True, exist_ok=True)

print("Required packages checked: pandas, pymongo, dnspython")
print(f"Dataset folder found: {DATA.name}")
print(f"MongoDB URI configured: {bool(os.getenv('MONGODB_URI'))}")

Required packages checked: pandas, pymongo, dnspython
Dataset folder found: medline_resit_dataset
MongoDB URI configured: True


## 2. Load MongoDB Source Data

This cell loads the nested JSONL patient service cases and the flat digital interaction file used for validation.

In [2]:
jsonl_path = DATA / "patient_service_cases.jsonl"
service_cases = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines() if line.strip()]

digital = pd.read_csv(DATA / "digital_interactions.csv")
patients = pd.read_csv(DATA / "patients.csv")
appointments = pd.read_csv(DATA / "appointments.csv")
feedback = pd.read_csv(DATA / "patient_feedback.csv")

load_summary = pd.DataFrame({
    "source": ["patient_service_cases.jsonl", "digital_interactions.csv", "patients.csv", "appointments.csv", "patient_feedback.csv"],
    "rows_or_documents": [len(service_cases), len(digital), len(patients), len(appointments), len(feedback)],
})
display(load_summary)

,source,rows_or_documents
0,patient_service_cases.jsonl,180
1,digital_interactions.csv,719
2,patients.csv,250
3,appointments.csv,700
4,patient_feedback.csv,350


## 3. Validate Nested Cases Against Digital Interactions

This cell checks that the nested JSONL service cases reconcile with the flat digital interaction events.

In [3]:
json_case_ids = {case["case_id"] for case in service_cases}
flat_case_ids = set(digital["case_id"])
json_event_count = sum(len(case.get("events", [])) for case in service_cases)
flat_event_count = len(digital)

consistency = pd.DataFrame([{
    "json_case_count": len(json_case_ids),
    "flat_case_count": len(flat_case_ids),
    "json_event_count": json_event_count,
    "flat_event_count": flat_event_count,
    "missing_from_json": len(flat_case_ids - json_case_ids),
    "missing_from_flat": len(json_case_ids - flat_case_ids),
}])
consistency.to_csv(TABLES / "mongodb_json_flat_consistency.csv", index=False)
display(consistency)

,json_case_count,flat_case_count,json_event_count,flat_event_count,missing_from_json,missing_from_flat
0,180,180,719,719,0,0


## 4. Service Case Summary

This cell summarises the service cases before importing them to MongoDB.

In [4]:
case_summary = pd.DataFrame([
    {
        "case_id": case["case_id"],
        "patient_id": case["patient_id"],
        "case_type": case["case_type"],
        "case_status": case["case_status"],
        "priority": case["priority"],
        "opened_date": case["opened_date"],
        "event_count": len(case.get("events", [])),
        "satisfaction_after_case": case.get("satisfaction_after_case"),
    }
    for case in service_cases
])
case_summary.to_csv(TABLES / "mongodb_service_case_summary.csv", index=False)
display(case_summary.head(10))

,case_id,patient_id,case_type,case_status,priority,opened_date,event_count,satisfaction_after_case
0,C00001,P0125,Referral query,Closed,Low,2026-01-02,6,2
1,C00002,P0187,Online consultation,Closed,High,2026-06-08,5,1
2,C00003,P0179,Prescription support,Escalated,Medium,2026-06-04,2,4
3,C00004,P0107,Community visit query,Open,Medium,2026-05-30,5,3
4,C00005,P0246,Prescription support,Closed,High,2026-06-27,2,1
5,C00006,P0246,Referral query,Closed,Medium,2026-06-14,4,4
6,C00007,P0099,Community visit query,Closed,Low,2026-05-08,5,2
7,C00008,P0049,Appointment issue,Closed,Low,2026-02-21,2,4
8,C00009,P0041,Referral query,Open,Low,2026-03-30,5,4
9,C00010,P0098,Prescription support,Escalated,Medium,2026-01-27,6,3


## 5. Nested Document Model Example

This cell builds one improved example document to show why MongoDB is suitable for semi-structured service cases.

In [5]:
patient_lookup = patients.set_index("patient_id").to_dict("index")
appointment_lookup = appointments.groupby("patient_id").head(3).groupby("patient_id").apply(lambda df: df.to_dict("records")).to_dict()
feedback_lookup = feedback.groupby("patient_id").head(2).groupby("patient_id").apply(lambda df: df.to_dict("records")).to_dict()

example_case = dict(service_cases[0])
patient_id = example_case["patient_id"]
example_case["patient_profile"] = patient_lookup.get(patient_id, {})
example_case["recent_appointments"] = appointment_lookup.get(patient_id, [])
example_case["recent_feedback"] = feedback_lookup.get(patient_id, [])

(MONGO_OUT / "enhanced_case_example.json").write_text(json.dumps(example_case, indent=2, default=str), encoding="utf-8")
print(json.dumps(example_case, indent=2, default=str)[:2500])

{
  "case_id": "C00001",
  "patient_id": "P0125",
  "case_type": "Referral query",
  "opened_date": "2026-01-02",
  "case_status": "Closed",
  "priority": "Low",
  "related_appointments": [
    "A00454",
    "A00418",
    "A00215"
  ],
  "events": [
    {
      "event_no": 1,
      "timestamp": "2026-01-02 11:30",
      "channel": "mobile_app",
      "event_type": "message_received",
      "note": "Status changed"
    },
    {
      "event_no": 2,
      "timestamp": "2026-01-03 10:30",
      "channel": "SMS",
      "event_type": "status_changed",
      "note": "Patient requested update"
    },
    {
      "event_no": 3,
      "timestamp": "2026-01-03 07:30",
      "channel": "web_form",
      "event_type": "patient_reply",
      "note": "Reminder sent"
    },
    {
      "event_no": 4,
      "timestamp": "2026-01-03 23:30",
      "channel": "mobile_app",
      "event_type": "message_received",
      "note": "Staff reviewed case"
    },
    {
      "event_no": 5,
      "timestamp": "202

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23352\1121473824.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  appointment_lookup = appointments.groupby("patient_id").head(3).groupby("patient_id").apply(lambda df: df.to_dict("records")).to_dict()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_23352\1121473824.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feedback_lookup = feedback.groupby("patient_id").head(2).gro

## 6. Connect To MongoDB Atlas And Import Documents

This cell connects to Atlas, creates/uses database `medline_resit`, creates/uses collection `patient_service_cases`, and imports the 180 service case documents.

In [6]:
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    print("MONGODB_URI is not configured. Set it privately in Colab before running live MongoDB cells.")
else:
    client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=10000)
    client.admin.command("ping")
    db = client["medline_resit"]
    collection = db["patient_service_cases"]

    collection.delete_many({})
    insert_result = collection.insert_many(service_cases)

    import_result = pd.DataFrame([{
        "database": "medline_resit",
        "collection": "patient_service_cases",
        "documents_inserted": len(insert_result.inserted_ids),
        "documents_in_collection": collection.count_documents({}),
    }])
    display(import_result)

,database,collection,documents_inserted,documents_in_collection
0,medline_resit,patient_service_cases,180,180


## 7. MongoDB CRUD Demonstration

This cell demonstrates create, read, update and delete using a separate test document so the imported dataset is not damaged.

In [7]:
if not os.getenv("MONGODB_URI"):
    print("Skipped live CRUD because MONGODB_URI is not configured.")
else:
    demo_doc = {
        "case_id": "C_TEST_DEMO",
        "patient_id": "P_TEST",
        "case_type": "Demo case",
        "opened_date": "2026-07-08",
        "case_status": "Open",
        "priority": "High",
        "related_appointments": [],
        "events": [{"event_no": 1, "event_type": "created", "channel": "web_form"}],
        "satisfaction_after_case": None,
    }
    collection.delete_many({"case_id": "C_TEST_DEMO"})
    collection.insert_one(demo_doc)
    read_before = collection.find_one({"case_id": "C_TEST_DEMO"}, {"_id": 0, "case_id": 1, "case_status": 1, "priority": 1})
    collection.update_one({"case_id": "C_TEST_DEMO"}, {"$set": {"case_status": "Escalated"}})
    read_after = collection.find_one({"case_id": "C_TEST_DEMO"}, {"_id": 0, "case_id": 1, "case_status": 1, "priority": 1})
    delete_result = collection.delete_one({"case_id": "C_TEST_DEMO"})

    crud_result = pd.DataFrame([
        {"operation": "CREATE/READ", "case_status": read_before["case_status"], "deleted_count": None},
        {"operation": "UPDATE/READ", "case_status": read_after["case_status"], "deleted_count": None},
        {"operation": "DELETE", "case_status": None, "deleted_count": delete_result.deleted_count},
    ])
    display(crud_result)

,operation,case_status,deleted_count
0,CREATE/READ,Open,NaN
1,UPDATE/READ,Escalated,NaN
2,DELETE,None,1.0


## 8. MongoDB Aggregation Pipelines

This cell defines aggregation pipelines for service-case monitoring.

In [8]:
aggregation_pipelines = {
    "cases_by_status_priority": [
        {"$group": {"_id": {"status": "$case_status", "priority": "$priority"}, "cases": {"$sum": 1}, "avg_satisfaction": {"$avg": "$satisfaction_after_case"}}},
        {"$sort": {"cases": -1}},
    ],
    "high_priority_open_cases": [
        {"$match": {"priority": "High", "case_status": {"$in": ["Open", "Escalated"]}}},
        {"$project": {"case_id": 1, "patient_id": 1, "case_type": 1, "opened_date": 1, "event_count": {"$size": "$events"}}},
        {"$sort": {"opened_date": 1}},
    ],
    "event_type_counts": [
        {"$unwind": "$events"},
        {"$group": {"_id": "$events.event_type", "events": {"$sum": 1}}},
        {"$sort": {"events": -1}},
    ],
    "channel_event_counts": [
        {"$unwind": "$events"},
        {"$group": {"_id": {"channel": "$events.channel", "event_type": "$events.event_type"}, "events": {"$sum": 1}}},
        {"$sort": {"events": -1}},
    ],
}

(MONGO_OUT / "aggregation_pipelines.json").write_text(json.dumps(aggregation_pipelines, indent=2), encoding="utf-8")
pipeline_list = pd.DataFrame({"pipeline_name": list(aggregation_pipelines.keys())})
display(pipeline_list)

,pipeline_name
0,cases_by_status_priority
1,high_priority_open_cases
2,event_type_counts
3,channel_event_counts


## 9. Run Aggregation Pipelines

This cell runs the MongoDB aggregation pipelines and saves the results.

In [9]:
if not os.getenv("MONGODB_URI"):
    print("Skipped live aggregations because MONGODB_URI is not configured.")
else:
    aggregation_results = {}
    for name, pipeline in aggregation_pipelines.items():
        aggregation_results[name] = list(collection.aggregate(pipeline))

    def clean_for_json(value):
        if isinstance(value, list):
            return [clean_for_json(item) for item in value]
        if isinstance(value, dict):
            return {key: clean_for_json(val) for key, val in value.items()}
        return str(value) if value.__class__.__name__ == "ObjectId" else value

    (MONGO_OUT / "aggregation_results.json").write_text(
        json.dumps(clean_for_json(aggregation_results), indent=2), encoding="utf-8"
    )

    display(pd.DataFrame(aggregation_results["cases_by_status_priority"]))
    display(pd.DataFrame(aggregation_results["event_type_counts"]))
    display(pd.DataFrame(aggregation_results["high_priority_open_cases"]).head(10))

,_id,cases,avg_satisfaction
0,"{'status': 'Closed', 'priority': 'Medium'}",40,3.150000
1,"{'status': 'Closed', 'priority': 'Low'}",39,3.205128
2,"{'status': 'Open', 'priority': 'Medium'}",25,2.800000
3,"{'status': 'Open', 'priority': 'Low'}",25,3.120000
4,"{'status': 'Closed', 'priority': 'High'}",16,2.937500
5,"{'status': 'Escalated', 'priority': 'Medium'}",14,3.142857
6,"{'status': 'Open', 'priority': 'High'}",12,2.416667
7,"{'status': 'Escalated', 'priority': 'Low'}",6,2.000000
8,"{'status': 'Escalated', 'priority': 'High'}",3,2.333333


,_id,events
0,status_changed,137
1,message_received,125
2,staff_note,123
3,reminder_sent,117
4,patient_reply,115
5,escalation,102


,_id,case_id,patient_id,case_type,opened_date,event_count
0,6a4e9c8c476ff14dc9ec0766,C00172,P0103,Referral query,2026-01-15,2
1,6a4e9c8c476ff14dc9ec0719,C00095,P0021,Online consultation,2026-01-24,4
2,6a4e9c8c476ff14dc9ec06fd,C00067,P0204,Prescription support,2026-02-21,3
3,6a4e9c8c476ff14dc9ec0708,C00078,P0048,Prescription support,2026-02-26,4
4,6a4e9c8c476ff14dc9ec0709,C00079,P0023,Online consultation,2026-02-26,4
5,6a4e9c8c476ff14dc9ec070f,C00085,P0128,Prescription support,2026-03-04,2
6,6a4e9c8c476ff14dc9ec0720,C00102,P0075,Online consultation,2026-03-30,3
7,6a4e9c8c476ff14dc9ec075d,C00163,P0146,Online consultation,2026-03-30,5
8,6a4e9c8c476ff14dc9ec06c8,C00014,P0208,Community visit query,2026-03-31,6
9,6a4e9c8c476ff14dc9ec06cd,C00019,P0050,Referral query,2026-04-08,3


## 10. Index Plan

This cell explains which indexes are needed and which query each index supports.

In [10]:
index_plan = pd.DataFrame([
    {"index_name": "case_id_unique", "fields": "case_id", "purpose": "Fast lookup of one service case"},
    {"index_name": "patient_id_index", "fields": "patient_id", "purpose": "Patient case history search"},
    {"index_name": "status_priority_opened", "fields": "case_status, priority, opened_date", "purpose": "Open/escalated dashboard queries"},
    {"index_name": "events_event_type", "fields": "events.event_type", "purpose": "Nested event-type analysis"},
    {"index_name": "events_channel", "fields": "events.channel", "purpose": "Nested communication-channel analysis"},
])
index_plan.to_csv(TABLES / "mongodb_index_plan.csv", index=False)
display(index_plan)

,index_name,fields,purpose
0,case_id_unique,case_id,Fast lookup of one service case
1,patient_id_index,patient_id,Patient case history search
2,status_priority_opened,"case_status, priority, opened_date",Open/escalated dashboard queries
3,events_event_type,events.event_type,Nested event-type analysis
4,events_channel,events.channel,Nested communication-channel analysis


## 11. Create MongoDB Indexes

This cell creates the indexes in Atlas and displays the index list.

In [11]:
if not os.getenv("MONGODB_URI"):
    print("Skipped index creation because MONGODB_URI is not configured.")
else:
    for item in collection.list_indexes():
        if item["name"] != "_id_":
            collection.drop_index(item["name"])

    collection.create_index([("case_id", ASCENDING)], name="case_id_unique", unique=True)
    collection.create_index([("patient_id", ASCENDING)], name="patient_id_index")
    collection.create_index([("case_status", ASCENDING), ("priority", ASCENDING), ("opened_date", DESCENDING)], name="status_priority_opened")
    collection.create_index([("events.event_type", ASCENDING)], name="events_event_type")
    collection.create_index([("events.channel", ASCENDING)], name="events_channel")

    index_list = pd.DataFrame([
        {"index_name": item["name"], "fields": str(item["key"])}
        for item in collection.list_indexes()
    ])
    display(index_list)

,index_name,fields
0,_id_,"SON([('_id', 1)])"
1,case_id_unique,"SON([('case_id', 1)])"
2,patient_id_index,"SON([('patient_id', 1)])"
3,status_priority_opened,"SON([('case_status', 1), ('priority', 1), ('op..."
4,events_event_type,"SON([('events.event_type', 1)])"
5,events_channel,"SON([('events.channel', 1)])"


## 12. Repeated Timing Tests

This cell runs each query five times and records timing evidence for optimisation.

In [12]:
query_workloads = {
    "open_high_priority_cases": {"priority": "High", "case_status": {"$in": ["Open", "Escalated"]}},
    "patient_case_history": {"patient_id": service_cases[0]["patient_id"]},
    "escalation_event_cases": {"events.event_type": "escalation"},
}
(MONGO_OUT / "query_workloads.json").write_text(json.dumps(query_workloads, indent=2), encoding="utf-8")

if not os.getenv("MONGODB_URI"):
    print("Skipped timing tests because MONGODB_URI is not configured.")
else:
    timing_rows = []
    for workload, query in query_workloads.items():
        for run in range(1, 6):
            start = time.perf_counter()
            list(collection.find(query).limit(50))
            duration_ms = (time.perf_counter() - start) * 1000
            timing_rows.append({"workload": workload, "run": run, "duration_ms": duration_ms})

    timing_df = pd.DataFrame(timing_rows)
    timing_summary = timing_df.groupby("workload", as_index=False).agg(avg_ms=("duration_ms", "mean"), min_ms=("duration_ms", "min"), max_ms=("duration_ms", "max"))
    timing_df.to_csv(TABLES / "mongodb_timing_results.csv", index=False)
    display(timing_summary.round(2))

,workload,avg_ms,min_ms,max_ms
0,escalation_event_cases,33.56,26.05,56.59
1,open_high_priority_cases,21.28,20.36,22.51
2,patient_case_history,20.94,18.81,25.48


## 13. Explain Plan Evidence

This cell records MongoDB explain-plan evidence so that index use can be checked.

In [13]:
if not os.getenv("MONGODB_URI"):
    print("Skipped explain plans because MONGODB_URI is not configured.")
else:
    explain_results = {}
    explain_rows = []
    for workload, query in query_workloads.items():
        explain_doc = collection.find(query).limit(50).explain()
        explain_results[workload] = explain_doc
        winning_plan = explain_doc.get("queryPlanner", {}).get("winningPlan", {})
        plan_text = json.dumps(winning_plan)
        explain_rows.append({
            "workload": workload,
            "uses_ixscan": "IXSCAN" in plan_text,
            "uses_collscan": "COLLSCAN" in plan_text,
            "winning_plan_stage": winning_plan.get("stage", "nested_plan"),
        })

    (MONGO_OUT / "explain_results.json").write_text(json.dumps(explain_results, indent=2, default=str), encoding="utf-8")
    explain_df = pd.DataFrame(explain_rows)
    explain_df.to_csv(TABLES / "mongodb_explain_notes.csv", index=False)
    display(explain_df)

,workload,uses_ixscan,uses_collscan,winning_plan_stage
0,open_high_priority_cases,True,False,LIMIT
1,patient_case_history,True,False,LIMIT
2,escalation_event_cases,True,False,LIMIT


## 14. Index Trade-Off Notes

This final evidence cell states the practical trade-off of indexing for the report and demo.

In [14]:
tradeoff_notes = pd.DataFrame([
    {"point": "Benefit", "explanation": "Indexes support faster read queries for case lookup, patient history and open high-priority cases."},
    {"point": "Cost", "explanation": "Indexes use extra storage and can slow inserts or updates because index entries must also be maintained."},
    {"point": "Decision", "explanation": "The selected indexes match repeated operational queries from the MedLine case study rather than indexing every field."},
])
tradeoff_notes.to_csv(TABLES / "optimisation_interpretation_points.csv", index=False)
display(tradeoff_notes)

,point,explanation
0,Benefit,Indexes support faster read queries for case l...
1,Cost,Indexes use extra storage and can slow inserts...
2,Decision,The selected indexes match repeated operationa...
